# conf 파라미터 튜닝 (YOLOv8L + MVP 15 클래스)

**실행 전 준비사항:**
1. 런타임 > 런타임 유형 변경 > **GPU (T4)** 선택
2. Google Drive에 `SampleVideo_Scenes` 폴더 업로드 (218개 mp4)
   - 경로: `내 드라이브/SampleVideo.Test/SampleVideo_Scenes/`

## 0. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("GPU 확인 완료!")

In [ ]:
# 패키지 설치
!pip install -q ultralytics
print("ultralytics 설치 완료!")

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# === 경로 설정 (필요시 수정) ===
DRIVE_SCENE_PATH = "/content/drive/MyDrive/SampleVideo.Test/SampleVideo_Scenes"

import os
scene_count = len([f for f in os.listdir(DRIVE_SCENE_PATH) if f.endswith('.mp4')])
print(f"Scene clips 확인: {scene_count}개")
assert scene_count == 218, f"218개여야 하는데 {scene_count}개 발견됨. 경로를 확인하세요."

## 1. conf 튜닝 실험 실행

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from tqdm.auto import tqdm

# =========================
# 기본 설정
# =========================
MODEL_PATH = "yolov8l.pt"
VIDEO_PATH = DRIVE_SCENE_PATH
DATA_YAML = "coco128.yaml"

# MVP 15 class ids
MVP_CLASS_IDS = [16, 26, 39, 41, 45, 53, 56, 57, 59, 60, 62, 63, 65, 67, 72]

# 고정 파라미터
FIXED_IOU = 0.7
FIXED_IMGSZ = 640

# conf 실험 범위: 0.15 ~ 0.70 (0.05 간격)
CONF_VALUES = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

# =========================
# 모델 로드
# =========================
print("Loading YOLOv8L model...")
model = YOLO(MODEL_PATH)

scene_files = sorted(Path(VIDEO_PATH).glob("*.mp4"))
print(f"Scene clips: {len(scene_files)}개")
print(f"실험 횟수: {len(CONF_VALUES)}개 conf 값")
print(f"총 추론 예상: {len(CONF_VALUES)} x {len(scene_files)} = {len(CONF_VALUES) * len(scene_files)}회")
print("=" * 50)

In [ ]:
rows = []

for exp_idx, conf_val in enumerate(CONF_VALUES, start=1):
    print(f"\n{'='*55}")
    print(f"  [{exp_idx}/{len(CONF_VALUES)}] conf={conf_val}, iou={FIXED_IOU}, imgsz={FIXED_IMGSZ}")
    print(f"{'='*55}")

    # --- COCO128 Validation ---
    print("  [1/2] COCO128 Validation...")
    val_start = time.time()
    metrics = model.val(
        data=DATA_YAML,
        imgsz=FIXED_IMGSZ,
        conf=conf_val,
        iou=FIXED_IOU,
        classes=MVP_CLASS_IDS,
        verbose=False
    )
    val_time = time.time() - val_start

    precision_arr = metrics.box.p
    recall_arr = metrics.box.r
    precision = float(np.nanmean(precision_arr)) if hasattr(precision_arr, "__len__") else float(precision_arr)
    recall = float(np.nanmean(recall_arr)) if hasattr(recall_arr, "__len__") else float(recall_arr)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"  Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f} ({val_time:.1f}s)")

    # --- SampleVideo_Scenes 추론 (stream=True) ---
    print(f"  [2/2] SampleVideo_Scenes 추론 ({len(scene_files)} clips)...")
    total_detected = 0
    start_time = time.time()

    pbar = tqdm(scene_files, desc=f"  conf={conf_val}", leave=True)
    for scene_file in pbar:
        results = model.predict(
            source=str(scene_file),
            conf=conf_val,
            iou=FIXED_IOU,
            imgsz=FIXED_IMGSZ,
            classes=MVP_CLASS_IDS,
            stream=True,
            verbose=False
        )
        for result in results:
            if result.boxes is not None:
                total_detected += len(result.boxes)
        pbar.set_postfix({"detected": total_detected})

    elapsed = time.time() - start_time
    det_per_sec = total_detected / elapsed if elapsed > 0 else 0

    print(f"  => 탐지: {total_detected}개 | 시간: {elapsed:.1f}s | {det_per_sec:.1f} det/s")

    rows.append({
        "실험": f"C{exp_idx}",
        "conf": conf_val,
        "iou": FIXED_IOU,
        "imgsz": FIXED_IMGSZ,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "탐지 수": total_detected,
        "소요 시간(s)": round(elapsed, 1),
        "탐지/sec": round(det_per_sec, 1)
    })

df = pd.DataFrame(rows)
print("\n" + "=" * 55)
print("  모든 실험 완료!")
print("=" * 55)

## 2. 결과 확인

In [ ]:
# 전체 결과 테이블
from IPython.display import display

# F1 최고값 강조
best_idx = df["F1"].idxmax()
best_row = df.loc[best_idx]

def highlight_best(row):
    if row.name == best_idx:
        return ['background-color: #90EE90; font-weight: bold'] * len(row)
    return [''] * len(row)

styled = df.style.apply(highlight_best, axis=1).format({
    'conf': '{:.2f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1': '{:.4f}',
    '탐지/sec': '{:.1f}'
})
display(styled)

print(f"\n★ Best F1 = {best_row['F1']} @ conf={best_row['conf']}")
print(f"  Precision={best_row['Precision']}, Recall={best_row['Recall']}")
print(f"  탐지 수={best_row['탐지 수']}, 소요 시간={best_row['소요 시간(s)']}s")

In [ ]:
# 시각화: Precision / Recall / F1 그래프
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- 왼쪽: P / R / F1 ---
ax1.plot(df['conf'], df['Precision'], 'b-o', label='Precision', linewidth=2)
ax1.plot(df['conf'], df['Recall'], 'r-s', label='Recall', linewidth=2)
ax1.plot(df['conf'], df['F1'], 'g-^', label='F1 Score', linewidth=2, markersize=8)

# Best F1 표시
ax1.axvline(x=best_row['conf'], color='green', linestyle='--', alpha=0.5)
ax1.annotate(f"Best F1={best_row['F1']:.4f}\nconf={best_row['conf']}",
             xy=(best_row['conf'], best_row['F1']),
             xytext=(best_row['conf']+0.08, best_row['F1']-0.05),
             arrowprops=dict(arrowstyle='->', color='green'),
             fontsize=10, color='green', fontweight='bold')

ax1.set_xlabel('Confidence Threshold', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Precision / Recall / F1 vs conf', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0.10, 0.75)
ax1.set_ylim(0, 1.05)

# --- 오른쪽: 탐지 수 ---
ax2.bar(df['conf'].astype(str), df['탐지 수'], color='steelblue', alpha=0.8)
ax2.set_xlabel('Confidence Threshold', fontsize=12)
ax2.set_ylabel('Total Detections', fontsize=12)
ax2.set_title('Detection Count vs conf', fontsize=13)
ax2.grid(True, alpha=0.3, axis='y')

for i, v in enumerate(df['탐지 수']):
    ax2.text(i, v + max(df['탐지 수'])*0.01, str(v), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('conf_tuning_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: conf_tuning_result.png")

## 3. 결과 저장 & 다운로드

In [ ]:
# CSV 저장
csv_path = "tune_conf_results.csv"
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"CSV 저장: {csv_path}")

# Drive에도 백업
drive_save_path = "/content/drive/MyDrive/SampleVideo.Test/tune_conf_results.csv"
df.to_csv(drive_save_path, index=False, encoding='utf-8-sig')
print(f"Drive 백업: {drive_save_path}")

# 그래프도 Drive 백업
import shutil
shutil.copy('conf_tuning_result.png', '/content/drive/MyDrive/SampleVideo.Test/conf_tuning_result.png')
print("그래프 Drive 백업 완료")

# 다운로드
from google.colab import files
files.download(csv_path)
files.download('conf_tuning_result.png')
print("\n다운로드 시작! 로컬에서 Project_Report.md에 결과를 반영하세요.")

In [ ]:
# Report용 마크다운 테이블 출력 (복사해서 Report에 붙여넣기)
print("=" * 60)
print("  아래 내용을 Project_Report.md에 붙여넣으세요")
print("=" * 60)
print()
print(df.to_markdown(index=False))
print()
print(f"\n★ Best F1 = {best_row['F1']} @ conf={best_row['conf']}")